In [ ]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, Tool  
from agents.mcp import MCPServerStdio
from openai import AsyncOpenAI
from datetime import datetime, date
import asyncio

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3-flash-preview", openai_client=gemini_client)

openai_model = "gpt-5-mini"
project_path = os.path.abspath(os.path.join(os.getcwd()))

files_params = {
    "command": "npx",
    "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        project_path
    ]
}

file_server = MCPServerStdio(params=files_params,client_session_timeout_seconds=60)

instructions = """
You are a certified financial analyst and expert of SEC EDGAR filings.
You are given a financial statement XBRL concept and best match against a predifined income statement line 
item for a company. The file with the mapping is income_statement_xbrl_mapping.py. 
The file already has the mapping for the most common concepts.
Use MCP Server tools to access income_statement_xbrl_mapping.py"""

async def verify_concept_mapping(instructions) -> Agent:
    instructions = instructions
    verify_concept_mapping = Agent(
        name="Verify Concept Mapping Agent",
        instructions=instructions,
        mcp_servers=[file_server],
        model=openai_model
    )
    return verify_concept_mapping
    
concept_agent = await verify_concept_mapping(instructions)

concept = "SellingAndMarketingExpense"
user_instructions = f"""
You are given the XBR concept {concept}.
Map the given XBRL concept to the best match income statement category line item in the predifined mapping file
test_predifined_xbrl_mapping.py.
Return the income statement line item from the file_server
test_predifined_xbrl_mapping.py. Return on the line item without extra verbage.
If the concept is already mapped, just return "Already mapped" without any extra verbage.
"""

#START MCP SERVER
await file_server.connect()

#RUN AGENT
with trace("Verify Concept Mapping Agent"):
    result = await Runner.run(concept_agent, user_instructions)


In [ ]:
print(result.final_output)

In [ ]:
import os
from edgar import Company , set_identity
from edgar.xbrl import XBRLS
from dotenv import load_dotenv

# Load environment variables
load_dotenv()


# Set SEC identity to avoid 403 blocks
sec_identity = os.getenv("SEC_ID")

set_identity(sec_identity)

company = Company("MSFT")

filing = company.latest("10-K")



In [ ]:
type(filing)

In [ ]:
from edgar import Company
from edgar import xbrl
from edgar.xbrl import XBRLS
import pandas as pd
import time

tickers_df = pd.read_csv("test_tickers.csv")
tickers_list = tickers_df['tickers'].tolist()


In [ ]:
all_inc_stmt_df = pd.DataFrame()

for ticker in tickers_list:
    print("extracting income statement for", ticker)
    company = Company(ticker)
    filing = company.get_filings(form="10-K", year = "2024", amendments=False).latest(1)
    
    try:
        xbrl = filing.xbrl()
        inc_stmt = xbrl.statements.income_statement()
    except Exception as e:
        print(f"Error extracting XBRL for {ticker}: {e}. Skipping to next ticker.")
        continue
    
    inc_stmt = xbrl.statements.income_statement()
    inc_stmt_df = inc_stmt.to_dataframe()
    inc_stmt_df = inc_stmt_df[inc_stmt_df['dimension'] != True]
    inc_stmt_df = inc_stmt_df.iloc[:,:3]
    
    time.sleep(0.25)
    all_inc_stmt_df = pd.concat([all_inc_stmt_df, inc_stmt_df], ignore_index=True)

all_inc_stmt_df.to_csv("all_inc_stmt_df.csv")


In [ ]:
all_inc_stmt_df.describe()

In [ ]:
all_unique_inc_stmt_df = all_inc_stmt_df.drop_duplicates(subset=['concept']).reset_index(drop=True)
all_unique_inc_stmt_df.describe()

In [ ]:
#print(all_unique_inc_stmt_df.iloc[:,0])
concepts_list = all_unique_inc_stmt_df.iloc[:,:1].values.tolist()

In [118]:
from openai import AsyncOpenAI, OpenAI
import asyncio
import json

from xbrl_mappings.income_statement_xbrl_mapping import INCOME_STATEMENT_MAPPING

OLLAMA_BASE_URL = "http://172.17.112.1:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
ollama_model = "deepseek-r1:8b"

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)
#openrouter_model = "stepfun/step-3.5-flash:free"
openrouter_model = "x-ai/grok-4.1-fast"
#### USE EXTRA BODY TO FORCE OPENROUTER TO USE CEREBRAS FOR GPT-OSS-120B
#openrouter_model = "openai/gpt-oss-120b"
# openrouter_extra_body={
#     "provider": {
#         "only": ["cerebras"],        # restrict to Cerebras
#         #"only": ["deepinfra"],        # restrict to Cerebras
#         "allow_fallbacks": False,    # fail instead of switching providers
#     },
# }  

income_statement_mapping_json = json.dumps(INCOME_STATEMENT_MAPPING)


In [ ]:
len(concepts_list)

In [ ]:
print(1275/12*0.0144)

In [119]:
import random

# Select 15 random concepts from concepts_list
random_concepts = random.sample(concepts_list, min(15, len(concepts_list)))


i = 0
grok_response = []
for concept in random_concepts:

    instructions = f""" You are a finance analyst and expert with SEC EDGAR filings and XBRL concepts.
    You are given a EDGAR financial statement XBRL concept and 
    a JSON file with predifined XBRL concepts to income statement line mappings. 
    Your job is to map the given XBRL concept to the best match income statement line item in the 
    predifined mapping file JSON. Return only the income statement line item that 
    best matches the given XBRL concept. No extra verbage.
    JSON file: {income_statement_mapping_json}
    XBRL concept: {concept}"""

    i += 1 
    
    response = openrouter_client.chat.completions.create(
        model=openrouter_model,
        messages=[{"role": "user", "content": instructions}],
        #NEED TO ADD EXTRA BODY TO FORCE OPENROUTER TO USE CEREBRAS FOR GPT-OSS-120B
        #extra_body=openrouter_extra_body
    )

    print("concept", concept[0], "response", response.choices[0].message.content)

    # Initialize oss_response on the first iteration and add each response to a list
    grok_response.append(response.choices[0].message.content)

    if i > 15:
        break



concept kkr_PolicyFeeIncome response revenue
concept exc_InterestExpenseToAffiliates response interest_expense
concept us-gaap_EarningsPerShareBasic response basic_eps
concept us-gaap_ResultsOfOperationsProductionOrLiftingCosts response cost_of_revenue
concept es_EnergyEfficiencyPrograms response other_operating_expenses
concept us-gaap_WaterProductionCosts response cost_of_revenue
concept us-gaap_IncomeLossFromDiscontinuedOperationsNetOfTaxAbstract response discontinued_operations
concept t_GoodwillImpairmentLossAndAssetsDisposedOfByMethodOtherThanSaleInPeriodOfDispositionLossOnDisposition response restructuring_charges
concept us-gaap_IncomeAmountsAttributableToReportingEntityDisclosuresAbstract response net_income_attributable_to_parent
concept ibkr_OccupancyDepreiationAndAmortization response depreciation_amortization
concept fitb_CardAndProcessingRevenue response revenue
concept gbx_AssetImpairmentDisposalAndExitCostsNet response restructuring_charges
concept us-gaap_InterestExpen

In [120]:
grok_response_dicts = [
    {"concept": concept[0], "income_statement_mapping": response}
    for concept, response in zip(random_concepts, grok_response)
]



In [115]:
print(step_response_dicts)

[{'concept': 'cl_InterestExpenseNet', 'income_statement_mapping': 'interest_expense'}, {'concept': 'coin_CryptoAssetImpairmentNet', 'income_statement_mapping': 'restructuring_charges'}, {'concept': 'us-gaap_EquityMethodInvestmentRealizedGainLossOnDisposal', 'income_statement_mapping': 'investment_gains_losses'}, {'concept': 'us-gaap_MarketRiskBenefitChangeInFairValueGainLoss', 'income_statement_mapping': 'other_operating_expenses'}, {'concept': 'pcg_LossFromCatastrophesGainFromInsuranceRecovery', 'income_statement_mapping': 'other_operating_expenses'}, {'concept': 'pld_GainLossOnForeignCurrencyDerivativeAndOtherGainsAndOtherIncomeExpenseNet', 'income_statement_mapping': 'other_nonoperating_income'}, {'concept': 'gbx_InterestAndForeignExchangeNet', 'income_statement_mapping': 'other_nonoperating_income'}, {'concept': 'oxy_TransportationAndGatheringExpense', 'income_statement_mapping': 'other_operating_expenses'}, {'concept': 'crh_EarningsPerShareOtherDisclosureAbstract', 'income_stateme

In [122]:
### SAVE TO FILE
# Convert oss_response (list) to newline-separated string and save to file
with open("grok_response_dicts.txt", "w") as f:
    f.write("\n".join(str(item) for item in grok_response_dicts))

In [ ]:

async def verify_concept_mapping(instructions) -> Agent:
    instructions = instructions,
    verify_concept_mapping = Agent(
        name="Verify Concept Mapping Agent",
        instructions=instructions

In [ ]:

i = 0
for concept in concepts_list:
    while i < 15:
        print(i,concept[0])
        i += 1
   


In [ ]:
import pandas as pd
tickers_df = pd.read_csv("test_tickers.csv")

### REMOVE DUPLICATES
unique_tickers_df = tickers_df.drop_duplicates(subset=['tickers']).reset_index(drop=True)
unique_tickers_df.to_csv("test_tickers.csv")

In [ ]:
unique_tickers_df.describe()

In [ ]:
#unique_tickers_df.describe()

# Check if 'NVDA' is in the dataframe
'NFLX' in unique_tickers_df['tickers'].str.upper().values

In [ ]:
def get_income_dataframe(ticker:str):
    c = Company(ticker)
    filings = c.get_filings(form="10-K").latest(5)
    xbs = XBRLS.from_filings(filings)
    income_statement = xbs.statements.income_statement()
    income_df = income_statement.to_dataframe()
    return income_df

In [ ]:
get_income_dataframe("AAPL")

In [ ]:
print(filings)

In [ ]:
inc_stmt = xbrl.statements.income_statement()

In [ ]:
inc_stmt_df = inc_stmt.to_dataframe
inc_stmt_df.to_csv("inc_stmt_df.csv")

In [ ]:
results = (xbrl.query()
            .by_concept("us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax")
           )

In [ ]:
print(results)

In [ ]:
print(statements)

In [ ]:
print(cashflow_stmt)

In [ ]:
df = income_stmt.to_df()
df.head()






In [ ]:
# Now try extraction again
cashflow_df = await extract_cash_flow(
    filing,
    TICKER,
    FILING_TYPE,
    YEAR,
    QUARTER,
    use_ai_fallback=USE_AI_FALLBACK
)

In [ ]:
print(result.final_output)


In [125]:
import duckdb
import ast

DB_PATH = "xbrl_mappings_multi.duckdb"
VALID_STATEMENT_TYPES = {"income", "balance", "cashflow"}

def import_grok_response_dicts(file_path: str, statement_type: str, db_path: str = DB_PATH) -> dict:
    """
    Import grok_response_dicts.txt into the ai_discovered_mappings table
    of xbrl_mappings_multi.duckdb.

    Each line in the file is a Python dict literal like:
        {'concept': 'kkr_PolicyFeeIncome', 'income_statement_mapping': 'revenue'}

    Args:
        file_path: Path to the response dicts text file.
        statement_type: One of 'income', 'balance', or 'cashflow'.

    Returns dict with counts: {'inserted': N, 'skipped': N, 'errors': N}
    """
    if statement_type not in VALID_STATEMENT_TYPES:
        raise ValueError(
            f"Invalid statement_type '{statement_type}'. "
            f"Must be one of: {', '.join(sorted(VALID_STATEMENT_TYPES))}"
        )

    con = duckdb.connect(db_path)
    inserted = 0
    skipped = 0
    errors = 0

    # Detect the mapping key from the first non-empty line
    mapping_key = None
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entry = ast.literal_eval(line)
            # Find the key that isn't 'concept'
            mapping_key = [k for k in entry if k != "concept"][0]
            break

    if not mapping_key:
        con.close()
        raise ValueError(f"Could not detect mapping key from {file_path}")

    print(f"Using mapping key: '{mapping_key}', statement_type: '{statement_type}'")

    with open(file_path, "r") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                entry = ast.literal_eval(line)
                concept = entry["concept"]
                field_name = entry[mapping_key]

                # Check if this mapping already exists
                existing = con.execute(
                    """SELECT id FROM ai_discovered_mappings
                       WHERE statement_type = ?
                         AND field_name = ?
                         AND concept = ?""",
                    [statement_type, field_name, concept],
                ).fetchone()

                if existing:
                    skipped += 1
                    print(f"  SKIP (duplicate): {concept} -> {field_name}")
                    continue

                con.execute(
                    """INSERT INTO ai_discovered_mappings
                       (statement_type, field_name, concept, confidence_score)
                       VALUES (?, ?, ?, 0.8)""",
                    [statement_type, field_name, concept],
                )
                inserted += 1
                print(f"  INSERT: {concept} -> {field_name}")

            except Exception as e:
                errors += 1
                print(f"  ERROR line {line_num}: {e}")

    con.close()
    summary = {"inserted": inserted, "skipped": skipped, "errors": errors}
    print(f"\nDone. {summary}")
    return summary

In [126]:
def concept_has_mapping(concept: str, db_path: str = DB_PATH) -> dict | None:
    """
    Check whether a concept already has an entry in xbrl_mappings_multi.duckdb.
    Searches both core_concept_mappings and ai_discovered_mappings.

    Args:
        concept: The XBRL concept string, e.g. 'us-gaap_EarningsPerShareBasic'

    Returns:
        A dict with the mapping info if found, e.g.:
            {'source': 'core', 'statement_type': 'income', 'field_name': 'basic_eps', 'concept': '...'}
        or None if the concept has no entry.
    """
    con = duckdb.connect(db_path, read_only=True)

    # 1) Check core_concept_mappings first (higher trust)
    row = con.execute(
        """SELECT statement_type, field_name, concept, priority
           FROM core_concept_mappings
           WHERE concept = ?
           ORDER BY priority ASC
           LIMIT 1""",
        [concept],
    ).fetchone()

    if row:
        con.close()
        return {
            "source": "core",
            "statement_type": row[0],
            "field_name": row[1],
            "concept": row[2],
            "priority": row[3],
        }

    # 2) Check ai_discovered_mappings
    row = con.execute(
        """SELECT statement_type, field_name, concept, confidence_score
           FROM ai_discovered_mappings
           WHERE concept = ?
           LIMIT 1""",
        [concept],
    ).fetchone()

    con.close()

    if row:
        return {
            "source": "ai_discovered",
            "statement_type": row[0],
            "field_name": row[1],
            "concept": row[2],
            "confidence_score": row[3],
        }

    return None

In [127]:
# --- Import the grok response dicts file ---
result = import_grok_response_dicts("grok_response_dicts.txt", statement_type="income")


Using mapping key: 'income_statement_mapping', statement_type: 'income'
  INSERT: kkr_PolicyFeeIncome -> revenue
  INSERT: exc_InterestExpenseToAffiliates -> interest_expense
  INSERT: us-gaap_EarningsPerShareBasic -> basic_eps
  INSERT: us-gaap_ResultsOfOperationsProductionOrLiftingCosts -> cost_of_revenue
  INSERT: es_EnergyEfficiencyPrograms -> other_operating_expenses
  INSERT: us-gaap_WaterProductionCosts -> cost_of_revenue
  INSERT: us-gaap_IncomeLossFromDiscontinuedOperationsNetOfTaxAbstract -> discontinued_operations
  INSERT: t_GoodwillImpairmentLossAndAssetsDisposedOfByMethodOtherThanSaleInPeriodOfDispositionLossOnDisposition -> restructuring_charges
  INSERT: us-gaap_IncomeAmountsAttributableToReportingEntityDisclosuresAbstract -> net_income_attributable_to_parent
  INSERT: ibkr_OccupancyDepreiationAndAmortization -> depreciation_amortization
  INSERT: fitb_CardAndProcessingRevenue -> revenue
  INSERT: gbx_AssetImpairmentDisposalAndExitCostsNet -> restructuring_charges
  INS

In [128]:

# --- Test the lookup function ---
print("\n--- Lookup tests ---")

# Should find in ai_discovered after import
test1 = concept_has_mapping("kkr_PolicyFeeIncome")
print(f"kkr_PolicyFeeIncome: {test1}")

# Should find in core_concept_mappings (us-gaap standard concept)
test2 = concept_has_mapping("EarningsPerShareBasic")
print(f"EarningsPerShareBasic: {test2}")

# Should return None (doesn't exist)
test3 = concept_has_mapping("NonExistentConcept_XYZ123")
print(f"NonExistentConcept_XYZ123: {test3}")


--- Lookup tests ---
kkr_PolicyFeeIncome: {'source': 'ai_discovered', 'statement_type': 'income', 'field_name': 'revenue', 'concept': 'kkr_PolicyFeeIncome', 'confidence_score': 0.800000011920929}
EarningsPerShareBasic: {'source': 'core', 'statement_type': 'income', 'field_name': 'basic_eps', 'concept': 'EarningsPerShareBasic', 'priority': 1}
NonExistentConcept_XYZ123: None
